## Method
1. Problem Formulation and Residual Learning

Harmonizing standard PET images to meet EARL reconstruction standards presents a unique challenge in deep learning: the source (Standard PET) and target (EARL PET) images are structurally and visually highly similar. A conventional image-to-image translation approach ($F(X) \rightarrow Y$) inevitably converges toward a local minimum of identity mapping, where the network learns to simply reproduce the input image without applying the required subtle filtering ($F(X) \approx X$).

To circumvent this issue, we reformulated the task as a residual learning problem. Rather than synthesizing the entire EARL image from scratch, the model is optimized to isolate and predict the exact difference map (the residual) between the standard and EARL reconstructions. This approach is mathematically expressed as:
$$\hat{R} ​≈ α(Y − X​)$$

Where $\hat{R}$ represents the predicted residual, $Y$ is the target EARL image, $X$ is the input standard PET image and $\alpha$ is a scalar amplification factor. By focusing on learning the residual, the model is encouraged to capture only the necessary adjustments needed to transform the standard PET into its EARL counterpart, effectively bypassing the identity mapping pitfall and enhancing the learning of subtle features that distinguish the two image types.

The finalized EARL image $\hat{Y}$ is then reconstructed by de-amplifying this prediction before adding it back to the original input image:$$\hat{Y} = X + \frac{\hat{R}_{amp}}{\alpha}$$

This design (i) forces the network to focus on the small, clinically relevant differences between acquisition/reconstruction standards and (ii) preserves the original macroscopic PET signal because the final EARL image is reconstructed by adding a (de-amplified) residual to the input image rather than synthesizing the full image from scratch.

2. U-Net based 2D-to-3D translation approach
The transformation from a standard PET space to the corresponding EARL image is fundamentally an operation of local filtering and Point Spread Function modification. By nature, this process is anatomy-agnostic and depends on local voxel interactions rather than macroscopic anatomical structures. We exploit this property afin de maximiser l'apport d'un ensemble d'entrainement limité., we opted for a patch-based UNet translation model using $64 \times 64$ crops, 

-  By decomposing volumes into individual slices, we can create larger twodimensional datasets with increased modes variability and diversity
- deconstruction of the global three-dimensional dataset into smaller, localized patches permet au model to solely focus on the translation task, allow une convergence with only few training samples
- which allows the network to learn local filtering patterns effectively  
- maintaining computational efficiency. 

The network takes as input a localized patch of the standard PET image with a minimal spatial context of neighbouring slices (5 in our case), and outputs the corresponding patch of the EARL image.

The global training objective is a weighted sum of two complementary terms that operate in the absolute SUV spaces and SSIM term as an additional constraint to ensure the preservation of the luminance, contrast, and structure of the original image:
$$\mathcal{L} = \lambda_{SUV} \cdot \mathcal 
{L}_{SUV} + \lambda_{SSIM} \cdot \mathcal{L}_{SSIM}$$



3. Non-Linear Data Normalization
Standardized Uptake Values (SUV) in PET imaging exhibit a highly skewed distribution, dominated by low-intensity background noise with sparse but intense physiological or tumoral "hot-spots" (e.g., brain, bladder, lesions). To stabilize the learning of the residual, we apply (percentile normalization) followed by a logarithmic transformation to the SUV values, which compresses the dynamic range and mitigates the influence of extreme values. The patches are scaled into [-1, 1] given the maximum log suv value in the training set, which is determined by the 99.9th percentile of the SUV distribution to avoid outliers dominating the scaling. This non-linear normalization strategy ensures that the network can effectively learn from both low and high-intensity regions without being biased towards the extreme values, thus enhancing the overall performance of the model in harmonizing PET images to EARL standards.